# 🦥 LLM Fine-Tuning with Unsloth
## A Beginner to Intermediate Guide

---

> **Unsloth** is an open-source Python library that lets you fine-tune large language models (LLMs) significantly faster and with much less memory consumption.

### 🎯 What You Will Learn

| Section | Topic | Level |
|---------|-------|-------|
| **1** | Installation & Library Imports | Beginner |
| **2** | Loading a Model (FastLanguageModel) | Beginner |
| **3** | Adding a LoRA Adapter | Beginner–Intermediate |
| **4** | Dataset Preparation & Prompt Template | Intermediate |
| **5** | Training with SFTTrainer | Intermediate |
| **6** | Inference (Testing the Model) | Intermediate |
| **7** | Saving the Model & GGUF Export | Intermediate |


---
## 📦 SECTION 1 — Installation & Library Imports
### (Slide 1: Imported Libraries / Modules)

Unsloth works on Google Colab or any local GPU environment.
The cell below installs all required packages.


In [ ]:
# ─────────────────────────────────────────────
# INSTALLATION — Run this cell once on first use
# ─────────────────────────────────────────────

# Install Unsloth for Colab (requires a GPU)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Supporting libraries
!pip install --no-deps trl peft accelerate bitsandbytes -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# ─────────────────────────────────────────────
# CORE LIBRARY IMPORTS
# ─────────────────────────────────────────────

# 🦥 UNSLOTH — Fast model loading and LoRA patching
from unsloth import FastLanguageModel

# 🤗 TRANSFORMERS — Tokenizer and base model infrastructure
from transformers import TrainingArguments

# 🎯 TRL (Transformer Reinforcement Learning) — Supervised fine-tuning loop
from trl import SFTTrainer

# 📊 DATASETS — Load datasets from HuggingFace Hub or local storage
from datasets import load_dataset

# 🔢 TORCH — GPU checks and tensor operations
import torch

print("✅ All libraries imported successfully!")
print(f"🖥️  GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮  GPU Name: {torch.cuda.get_device_name(0)}")

✅ All libraries imported successfully!
🖥️  GPU Available: True
🎮  GPU Name: Tesla T4


### 📖 Library Descriptions

| Library | Purpose |
|---------|--------|
| `unsloth.FastLanguageModel` | Loads LLMs 2x faster and uses ~60% less VRAM |
| `transformers.TrainingArguments` | Configures training hyperparameters: batch size, learning rate, epochs |
| `trl.SFTTrainer` | Manages the Supervised Fine-Tuning (SFT) training loop |
| `datasets.load_dataset` | Loads datasets from HuggingFace Hub or local files |
| `torch` | GPU memory management and tensor computation |


---
## 🤖 SECTION 2 — Loading the Model
### (Slide 2: Code Snippet — Model Loading)

`FastLanguageModel.from_pretrained()` loads a model from HuggingFace using Unsloth's
optimised kernels. Compared to standard `transformers`, it is **~2x faster** and uses significantly less memory.


In [ ]:
# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

MODEL_NAME    = "unsloth/Llama-3.2-1B-Instruct"  # Model to use (from HuggingFace Hub)
MAX_SEQ_LEN   = 2048    # Maximum token length — longer = more VRAM
DTYPE         = None    # None → auto-detect (Float16 / BFloat16)
LOAD_IN_4BIT  = True    # 4-bit quantization → reduces VRAM usage by ~75%

print(f"📌 Model : {MODEL_NAME}")
print(f"📏 Max Sequence Length: {MAX_SEQ_LEN} tokens")
print(f"🔢 4-bit Quantization : {LOAD_IN_4BIT}")

📌 Model : unsloth/Llama-3.2-1B-Instruct
📏 Max Sequence Length: 2048 tokens
🔢 4-bit Quantization : True


In [ ]:
# ─────────────────────────────────────────────
# LOAD MODEL & TOKENIZER
# ─────────────────────────────────────────────

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,      # HuggingFace model identifier
    max_seq_length = MAX_SEQ_LEN,     # Context window size
    dtype          = DTYPE,           # Compute dtype (auto)
    load_in_4bit   = LOAD_IN_4BIT,    # Load in 4-bit for QLoRA
)

print("✅ Model and Tokenizer loaded successfully!")
print(f"🧠 Model Type: {type(model).__name__}")

# Show total parameter count
total_params = sum(p.numel() for p in model.parameters())
print(f"📊 Total Parameters: {total_params / 1e9:.2f} Billion")

==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


✅ Model and Tokenizer loaded successfully!
🧠 Model Type: LlamaForCausalLM
📊 Total Parameters: 0.77 Billion


### 🔍 Parameter Explanations

```python
model_name     → Which LLM to use (Llama, Mistral, Phi, Gemma, ...)
max_seq_length → How many tokens the model can process at once
dtype          → Numerical precision — None lets Unsloth choose automatically
load_in_4bit   → QLoRA technique: compresses the model to 4-bit, saving VRAM
```

> 💡 **Tip:** On GPUs with less than 8 GB VRAM, `load_in_4bit=True` is essentially required!


---
## 🎛️ SECTION 3 — Adding a LoRA Adapter
### (Slide 3: Code Snippet — LoRA / PEFT)

**LoRA (Low-Rank Adaptation)** is a technique that inserts small trainable "adapter" layers
into the model instead of updating all original weights.

```
Full Fine-Tuning  →  ~7 Billion parameters updated on a 7B model
LoRA Fine-Tuning  →  only ~40 Million parameters updated  (~0.5%!)
```


In [ ]:
# ─────────────────────────────────────────────
# ADD LORA ADAPTER
# ─────────────────────────────────────────────

model = FastLanguageModel.get_peft_model(
    model,

    r = 16,                    # LoRA rank — lower = fewer params, higher = more capacity

    target_modules = [         # Which layers receive the LoRA adapter
        "q_proj",              # Query projection — attention mechanism
        "k_proj",              # Key projection
        "v_proj",              # Value projection
        "o_proj",              # Output projection
        "gate_proj",           # Feed-forward gate
        "up_proj",             # Feed-forward up layer
        "down_proj",           # Feed-forward down layer
    ],

    lora_alpha    = 16,        # LoRA scaling factor (usually set equal to r)
    lora_dropout  = 0,         # Dropout — 0 enables Unsloth's optimised mode
    bias          = "none",    # Bias training — "none" is the most efficient option
    use_gradient_checkpointing = "unsloth",  # Reduces VRAM by an additional ~30%
    random_state  = 42,        # Fixed seed for reproducibility
    use_rslora    = False,      # Rank-Stabilized LoRA — advanced usage
    loftq_config  = None,      # LoftQ quantization — advanced usage
)

print("✅ LoRA adapter added!")

# Show trainable vs total parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"🎯 Trainable Parameters : {trainable:,}  ({100 * trainable / total:.2f}%)")
print(f"📊 Total Parameters     : {total:,}")

✅ LoRA adapter added!
🎯 Trainable Parameters : 11,272,192  (1.43%)
📊 Total Parameters     : 785,713,152


### 🔍 LoRA Parameter Explanations

| Parameter | Description | Typical Value |
|-----------|-------------|---------------|
| `r` | Rank of the LoRA matrices — higher rank = more learning capacity | 8 – 64 |
| `lora_alpha` | Scaling factor for the learned weights | same as `r` |
| `target_modules` | Which transformer layers receive adapters | q/k/v/o_proj |
| `lora_dropout` | Overfitting prevention — 0 enables Unsloth's fast path | 0 – 0.1 |
| `use_gradient_checkpointing` | Recomputes intermediate activations to save VRAM | `"unsloth"` |


---
## 📝 SECTION 4 — Dataset Preparation & Prompt Template
### (Slide 4: Code Snippet — Data & Prompt Template)


In [ ]:
# ─────────────────────────────────────────────
# ALPACA PROMPT TEMPLATE  (Instruction Following)
# ─────────────────────────────────────────────

# This template shows the model examples in the format: task + context + response
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# EOS token — signals to the model that the answer is complete
EOS_TOKEN = tokenizer.eos_token
print(f"📌 EOS Token: {EOS_TOKEN}")

# ─────────────────────────────────────────────
# FORMATTING FUNCTION
# ─────────────────────────────────────────────

def formatting_prompts_func(examples):
    """
    Converts raw dataset rows into the Alpaca prompt format.

    Each example contains:
      - instruction : the task description given to the model
      - input       : optional additional context (may be empty)
      - output      : the expected response
    """
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []

    for instruction, inp, output in zip(instructions, inputs, outputs):
        # Fill the template and append the EOS token (required for training)
        text = alpaca_prompt.format(instruction, inp, output) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}  # Return as a new "text" column in the dataset

print("✅ Prompt template and formatting function are ready!")

📌 EOS Token: <|eot_id|>
✅ Prompt template and formatting function are ready!


In [ ]:
# ─────────────────────────────────────────────
# LOAD DATASET
# ─────────────────────────────────────────────

# Alpaca-format sample dataset from HuggingFace Hub
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

print(f"📚 Dataset size : {len(dataset):,} examples")
print(f"📋 Columns      : {dataset.column_names}")

# Preview the first example
print("\n🔍 First Example:")
print(f"  Instruction : {dataset[0]['instruction'][:80]}...")
print(f"  Input       : {dataset[0]['input'][:80] or '(empty)'}")
print(f"  Output      : {dataset[0]['output'][:80]}...")

📚 Dataset size : 51,760 examples
📋 Columns      : ['output', 'input', 'instruction']

🔍 First Example:
  Instruction : Give three tips for staying healthy....
  Input       : (empty)
  Output      : 1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a v...


In [ ]:
# ─────────────────────────────────────────────
# FORMAT THE DATASET
# ─────────────────────────────────────────────

dataset = dataset.map(
    formatting_prompts_func,
    batched = True,   # Process in batches — faster
)

print("✅ Dataset formatted!")
print(f"📋 Updated columns: {dataset.column_names}")

# Preview the formatted first example
print("\n🔍 Formatted First Example (first 300 characters):")
print(dataset[0]["text"][:300])

✅ Dataset formatted!
📋 Updated columns: ['output', 'input', 'instruction', 'text']

🔍 Formatted First Example (first 300 characters):
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Give three tips for staying healthy.

### Input:


### Response:
1. Eat a balanced and nutritious diet: Make sure your meals


---
## 🏋️ SECTION 5 — Training Configuration with SFTTrainer

**SFTTrainer** (Supervised Fine-Tuning Trainer) is TRL's training loop.
It integrates natively with Unsloth for optimised speed.


In [ ]:
# ─────────────────────────────────────────────
# TRAINING ARGUMENTS
# ─────────────────────────────────────────────

training_args = TrainingArguments(
    per_device_train_batch_size   = 2,       # Examples processed per GPU at once
    gradient_accumulation_steps   = 4,       # Update every 4 steps → effective batch = 2×4 = 8
    warmup_steps                  = 5,       # Gradually ramp up the learning rate at the start
    max_steps                     = 60,      # Total training steps (low for quick experiments)
    learning_rate                 = 2e-4,    # Learning rate — standard for LoRA fine-tuning
    fp16                          = not torch.cuda.is_bf16_supported(),  # Auto-select precision
    bf16                          = torch.cuda.is_bf16_supported(),      # Better on Ampere GPUs
    logging_steps                 = 1,       # Log metrics every step
    optim                         = "adamw_8bit",  # 8-bit AdamW — saves VRAM
    weight_decay                  = 0.01,    # Regularisation to prevent overfitting
    lr_scheduler_type             = "linear",     # How the learning rate decays
    seed                          = 42,      # Fixed seed for reproducibility
    output_dir                    = "outputs",    # Directory for checkpoints
)

print("✅ Training arguments configured!")
print(f"📊 Effective Batch Size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"⚡ Precision           : {'BF16' if training_args.bf16 else 'FP16'}")

✅ Training arguments configured!
📊 Effective Batch Size: 8
⚡ Precision           : FP16


In [ ]:
# ─────────────────────────────────────────────
# CREATE TRAINER
# ─────────────────────────────────────────────

trainer = SFTTrainer(
    model              = model,           # LoRA-patched Unsloth model
    tokenizer          = tokenizer,       # Model's tokenizer
    train_dataset      = dataset,         # Training data
    dataset_text_field = "text",          # Which column to use from the dataset
    max_seq_length     = MAX_SEQ_LEN,     # Maximum token length
    dataset_num_proc   = 2,               # Parallel data processing workers
    packing            = False,           # Pack short examples together — speed vs accuracy
    args               = training_args,   # Training arguments defined above
)

print("✅ Trainer is ready!")

✅ Trainer is ready!


In [ ]:
# ─────────────────────────────────────────────
# GPU STATUS BEFORE TRAINING
# ─────────────────────────────────────────────

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory       = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"🎮 GPU              : {gpu_stats.name}")
print(f"💾 Total VRAM       : {max_memory} GB")
print(f"📈 Currently in use : {start_gpu_memory} GB")

# ─────────────────────────────────────────────
# START TRAINING
# ─────────────────────────────────────────────

print("\n🚀 Starting training...")
trainer_stats = trainer.train()

# ─────────────────────────────────────────────
# POST-TRAINING STATISTICS
# ─────────────────────────────────────────────

used_memory          = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)

print(f"\n✅ Training complete!")
print(f"⏱️  Duration          : {round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes")
print(f"💾 Peak VRAM         : {used_memory} GB")
print(f"🎯 Extra VRAM for LoRA: {used_memory_for_lora} GB")

🎮 GPU              : Tesla T4
💾 Total VRAM       : 14.563 GB
📈 Currently in use : 5.529 GB

🚀 Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51,760 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
1,1.973480
2,2.132393
3,2.200871
4,2.192150
5,1.686589
6,1.662553
7,1.557111
8,1.612103
9,1.394307
10,1.396671



✅ Training complete!
⏱️  Duration          : 1.34 minutes
💾 Peak VRAM         : 6.037 GB
🎯 Extra VRAM for LoRA: 0.508 GB


---
## 🧪 SECTION 6 — Inference (Testing the Fine-Tuned Model)

After fine-tuning, switch to **inference mode** before generating text.
This enables Unsloth's optimised kernels and makes generation ~2x faster.


In [ ]:
# ─────────────────────────────────────────────
# SWITCH TO INFERENCE MODE
# ─────────────────────────────────────────────

# FastLanguageModel.for_inference(model)  # Enables Unsloth's 2x faster inference

# ─────────────────────────────────────────────
# PREPARE TEST PROMPT
# ─────────────────────────────────────────────

inputs = tokenizer(
    [
        alpaca_prompt.format(
            "Explain the concept of AI.",  # Simplified Instruction
            "",                                                     # Input / context left blank
            "",                                                       # Response left blank
        )
    ],
    return_tensors = "pt",  # Return as PyTorch tensors
).to("cuda")               # Move to GPU

print("✅ Input ready!")
print(f"📏 Input length: {inputs['input_ids'].shape[1]} tokens")

✅ Input ready!
📏 Input length: 44 tokens


In [ ]:
# ─────────────────────────────────────────────
# ACTIVATE INFERENCE MODE & GENERATE
# ─────────────────────────────────────────────

model = FastLanguageModel.for_inference(model)

outputs = model.generate(
    **inputs,
    max_new_tokens = 64,
    use_cache      = False  # Set to False to resolve the broadcast shape mismatch error
)

# ─────────────────────────────────────────────
# DECODE OUTPUT
# ─────────────────────────────────────────────

response = tokenizer.batch_decode(outputs)

print("\n🤖 Model Response:")
print("=" * 50)
response_text = response[0].split("### Response:")[-1].strip()
print(response_text)

Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



🤖 Model Response:
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that would normally require human intelligence, such as learning, problem-solving, and decision-making. The term "artificial intelligence" was first coined in 1956 by John McCarthy, a computer scientist who proposed the field of AI. AI


In [ ]:
# ─────────────────────────────────────────────
# STREAMING INFERENCE — Real-time token output
# ─────────────────────────────────────────────

from transformers import TextStreamer

# TextStreamer prints tokens to the screen as they are generated (ChatGPT-style)
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

inputs2 = tokenizer(
    [
        alpaca_prompt.format(
            "Explain what Machine Learning is in 2 sentences.",  # Instruction
            "",                                                   # No context
            "",                                                   # Response left blank
        )
    ],
    return_tensors = "pt",
).to("cuda")

print("🤖 Model (streaming response):")
print("=" * 50)

_ = model.generate(
    **inputs2,
    streamer       = text_streamer,   # Streaming output
    max_new_tokens = 128,
    temperature    = 1.0,
    use_cache      = False           # Set to False to resolve the broadcast shape mismatch error
)

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Model (streaming response):
Machine learning 

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


is a subset of artificial intelligence that uses algorithms to allow computers to learn and improve without explicit programming. By training and refining machine learning models with large amounts of data, we can effectively make decisions and predictions, often without being explicitly told what to do.<|eot_id|>


---
## 💾 SECTION 7 — Saving the Model & GGUF Export

After fine-tuning you can save the model in several formats depending on your use case.


In [ ]:
# ─────────────────────────────────────────────
# METHOD 1: Save LoRA Adapter Only
# ─────────────────────────────────────────────

model.save_pretrained("lora_model")      # Only the LoRA weights (~a few MB)
tokenizer.save_pretrained("lora_model")  # Save the tokenizer alongside

print("✅ LoRA adapter saved to 'lora_model/'")
print("📁 Size: Just the LoRA weights (a few MB)")

# ─────────────────────────────────────────────
# METHOD 2: Save Full Merged Model
# ─────────────────────────────────────────────

# Merge the LoRA weights into the base model and save as a single file
model.save_pretrained_merged(
    "full_model",                   # Output directory
    tokenizer,
    save_method = "merged_16bit"    # 16-bit merged model
)

print("\n✅ Merged model saved to 'full_model/'")

✅ LoRA adapter saved to 'lora_model/'
📁 Size: Just the LoRA weights (a few MB)


config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:16<00:00, 16.37s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:33<00:00, 33.62s/it]


Unsloth: Merge process complete. Saved to `/content/full_model`

✅ Merged model saved to 'full_model/'


In [ ]:
# ─────────────────────────────────────────────
# METHOD 3: GGUF Format — for Ollama & llama.cpp
# ─────────────────────────────────────────────

# GGUF is an optimised format for local inference (Ollama, LM Studio, etc.)
model.save_pretrained_gguf(
    "gguf_model",                        # Output directory
    tokenizer,
    quantization_method = "q4_k_m"       # 4-bit quantization — good quality/speed trade-off
)

print("✅ GGUF model saved to 'gguf_model/'")
print("🦙 Run locally with Ollama:")
print("   ollama run gguf_model/model.gguf")

# ─────────────────────────────────────────────
# METHOD 4: Push to HuggingFace Hub
# ─────────────────────────────────────────────

# Uncomment the lines below to share your model on HuggingFace Hub
# model.push_to_hub("your_username/your_model_name", token="HF_TOKEN")
# tokenizer.push_to_hub("your_username/your_model_name", token="HF_TOKEN")

print("\n💡 To upload to HuggingFace Hub, uncomment the lines above and add your HF_TOKEN.")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:54<00:00, 114.79s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:56<00:00, 56.93s/it]


Unsloth: Merge process complete. Saved to `/content/gguf_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['gguf_model_gguf/llama-3.2-1b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['gguf_model_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model gguf_model_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to gguf_model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f gguf_model_gguf/Modelfile
✅ GGUF model saved to 'gguf_model/'
🦙 Run locally with Ollama:
   ollama run gguf_model/model.gguf

💡 To upload to HuggingFace Hub, uncomment the lines above and add your HF_TOKEN.


---
## 📊 SECTION 8 — BONUS: Unsloth Comparison Table


In [ ]:
# ─────────────────────────────────────────────
# UNSLOTH vs STANDARD APPROACH COMPARISON
# ─────────────────────────────────────────────

import pandas as pd

comparison_data = {
    "Feature": [
        "Training Speed",
        "VRAM Usage",
        "Required GPU VRAM",
        "Llama 3.2 1B Fine-tune",
        "Llama 3.1 8B Fine-tune",
        "Supported Models",
    ],
    "Standard HuggingFace": [
        "1x (baseline)",
        "100%",
        "~16 GB",
        "~14 minutes",
        "~60 minutes",
        "All models",
    ],
    "Unsloth + QLoRA": [
        "~2x faster 🚀",
        "40% less ✅",
        "~5–6 GB 💪",
        "~7 minutes",
        "~30 minutes",
        "Llama, Mistral, Phi, Gemma…",
    ],
}

df = pd.DataFrame(comparison_data)
print("📊 Unsloth vs Standard HuggingFace Fine-Tuning:\n")
print(df.to_string(index=False))

📊 Unsloth vs Standard HuggingFace Fine-Tuning:

               Feature Standard HuggingFace             Unsloth + QLoRA
        Training Speed        1x (baseline)                ~2x faster 🚀
            VRAM Usage                 100%                  40% less ✅
     Required GPU VRAM               ~16 GB                   ~5–6 GB 💪
Llama 3.2 1B Fine-tune          ~14 minutes                  ~7 minutes
Llama 3.1 8B Fine-tune          ~60 minutes                 ~30 minutes
      Supported Models           All models Llama, Mistral, Phi, Gemma…


In [ ]:
# ─────────────────────────────────────────────
# SUPPORTED MODELS
# ─────────────────────────────────────────────

supported_models = {
    "Llama Family": [
        "unsloth/Llama-3.2-1B-Instruct",
        "unsloth/Llama-3.2-3B-Instruct",
        "unsloth/Meta-Llama-3.1-8B",
        "unsloth/Meta-Llama-3.1-70B",
    ],
    "Mistral Family": [
        "unsloth/mistral-7b-v0.3",
        "unsloth/Mistral-Nemo-Instruct-2407",
    ],
    "Google Models": [
        "unsloth/gemma-2-9b",
        "unsloth/gemma-2-27b",
    ],
    "Microsoft Phi": [
        "unsloth/Phi-3.5-mini-instruct",
        "unsloth/phi-4",
    ],
    "Qwen Family": [
        "unsloth/Qwen2.5-7B-Instruct",
        "unsloth/Qwen2.5-72B-Instruct",
    ],
}

print("🦥 Models Supported by Unsloth:\n")
for family, models in supported_models.items():
    print(f"  📌 {family}:")
    for m in models:
        print(f"     - {m}")
    print()

🦥 Models Supported by Unsloth:

  📌 Llama Family:
     - unsloth/Llama-3.2-1B-Instruct
     - unsloth/Llama-3.2-3B-Instruct
     - unsloth/Meta-Llama-3.1-8B
     - unsloth/Meta-Llama-3.1-70B

  📌 Mistral Family:
     - unsloth/mistral-7b-v0.3
     - unsloth/Mistral-Nemo-Instruct-2407

  📌 Google Models:
     - unsloth/gemma-2-9b
     - unsloth/gemma-2-27b

  📌 Microsoft Phi:
     - unsloth/Phi-3.5-mini-instruct
     - unsloth/phi-4

  📌 Qwen Family:
     - unsloth/Qwen2.5-7B-Instruct
     - unsloth/Qwen2.5-72B-Instruct



---
## 🎯 SUMMARY — Unsloth Fine-Tuning Pipeline

```
1. pip install unsloth
        ↓
2. FastLanguageModel.from_pretrained()   ← Load model (4-bit)
        ↓
3. FastLanguageModel.get_peft_model()    ← Attach LoRA adapter
        ↓
4. Prepare dataset + prompt template     ← Format your data
        ↓
5. SFTTrainer.train()                    ← Fine-tune!
        ↓
6. FastLanguageModel.for_inference()     ← Test the model
        ↓
7. save_pretrained() / GGUF export       ← Save & deploy
```

### 🔗 Useful Resources

- 📖 Unsloth GitHub  : https://github.com/unslothai/unsloth
- 🤗 HuggingFace Hub : https://huggingface.co/unsloth
- 📓 Example Notebooks: https://github.com/unslothai/unsloth/tree/main/notebooks
- 💬 Discord         : https://discord.gg/unsloth
